# Import all required libraries

In [8]:
pip install -q rapidfuzz

In [9]:
import numpy as np
import pandas as pd
from datetime import timedelta
from rapidfuzz import fuzz

# Data Ingestion

In [10]:
# Loading excel files of Ledger and Bank statement

ledger_df = pd.read_excel('/content/Ledger_InputData (1).xlsx')
bank_df = pd.read_excel('/content/Statement_InputData (1).xlsx')

In [11]:
# Standardizing column names
ledger_df.columns = ledger_df.columns.str.strip()
bank_df.columns = bank_df.columns.str.strip()

print("Ledger Shape:", ledger_df.shape)
print("Bank Shape:", bank_df.shape)

Ledger Shape: (2977, 10)
Bank Shape: (250, 11)


In [46]:
from rapidfuzz import fuzz

s1 = "ABCDE"
s2 = "Ab"

# Simple ratio
score = fuzz.ratio(s1, s2)

print("Similarity Score:", score)

Similarity Score: 28.57142857142857


# EDA

In [12]:
ledger_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2977 entries, 0 to 2976
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Doc Type            2938 non-null   object        
 1   Posting Dt          2977 non-null   datetime64[ns]
 2   Doc No              2977 non-null   int64         
 3   Cheque No           14 non-null     float64       
 4   Description         1268 non-null   object        
 5   Particulars         832 non-null    object        
 6   Debit-Credit        2977 non-null   float64       
 7   Bank                2946 non-null   object        
 8   GL A/C              2969 non-null   float64       
 9   GL A/c Description  2977 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(5)
memory usage: 232.7+ KB


In [13]:
bank_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          250 non-null    datetime64[ns]
 1   Unique ID     215 non-null    object        
 2   Bank Ref      249 non-null    object        
 3   Cust Ref      249 non-null    object        
 4   Narration     249 non-null    object        
 5   Narration 2   249 non-null    object        
 6   Opening Flag  1 non-null      object        
 7   Credit-Debit  250 non-null    float64       
 8   BANK          250 non-null    object        
 9   Credit        34 non-null     float64       
 10  Debit         215 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 21.6+ KB


In [14]:
# Create unified amount columns
ledger_df['Amount'] = ledger_df['Debit-Credit'].fillna(0)
bank_df['Amount'] = bank_df['Credit-Debit'].fillna(0)

In [15]:
#  Understanding Debit/Credit in Ledger & Statement
print("\n Debits & Credits in LEDGER")
print("Ledger Debit Total:", ledger_df[ledger_df['Amount'] > 0]['Amount'].sum())
print("Ledger Credit Total:", ledger_df[ledger_df['Amount'] < 0]['Amount'].sum())
print("Ledger balances against Debit/Credit:", ledger_df['Amount'].sum())


print("\n Debits & Credits in STATEMENT")
print("Bank Debit Total:", bank_df[bank_df['Amount'] < 0]['Amount'].sum())
print("Bank Credit Total:", bank_df[bank_df['Amount'] > 0]['Amount'].sum())
print("Bank balances against Debit/Credit:", bank_df['Amount'].sum())

print("\n Overall Difference:", ledger_df['Amount'].sum() - bank_df['Amount'].sum())


 Debits & Credits in LEDGER
Ledger Debit Total: 116410182.16
Ledger Credit Total: -123096062.35779999
Ledger balances against Debit/Credit: -6685880.19780001

 Debits & Credits in STATEMENT
Bank Debit Total: -32271013.099999998
Bank Credit Total: 31409973.31
Bank balances against Debit/Credit: -861039.7899999991

 Overall Difference: -5824840.407800011


In [16]:
# Normalize narration text
ledger_df['Narration'] = ledger_df['Description'].astype(str).str.lower().str.strip()
bank_df['Narration'] = (bank_df['Narration'].astype(str) + " " + bank_df['Narration 2'].astype(str)).str.lower().str.strip()

In [17]:
# Duplicates Check
print("Ledger Duplicates:", ledger_df.duplicated().sum())
print("Bank Duplicates:", bank_df.duplicated().sum())

Ledger Duplicates: 378
Bank Duplicates: 0


In [18]:
# Duplicates data
duplicate_rows = ledger_df[ledger_df.duplicated(keep=False)]

# Sort them so the identical rows are next to each other
print(duplicate_rows.sort_values(by=list(ledger_df.columns)).head(10))

     Doc Type Posting Dt  Doc No  Cheque No Description  \
1910       KZ 2021-01-12  601017        NaN      Salary   
1911       KZ 2021-01-12  601017        NaN      Salary   
33         KZ 2021-01-18  601038        NaN      Salary   
34         KZ 2021-01-18  601038        NaN      Salary   
35         KZ 2021-01-18  601038        NaN      Salary   
38         KZ 2021-01-18  601038        NaN      Salary   
1683       KZ 2021-01-18  601038        NaN      Salary   
30         KZ 2021-01-18  601038        NaN      Salary   
37         KZ 2021-01-18  601038        NaN      Salary   
848        KZ 2021-01-18  601038        NaN      Salary   

                        Particulars  Debit-Credit                        Bank  \
1910  PAID AGAINST EE EXP REIMBURSE        -145.0  WELLS FARGO BANK-121000248   
1911  PAID AGAINST EE EXP REIMBURSE        -145.0  WELLS FARGO BANK-121000248   
33    PAID AGAINST EE EXPENSE REIMB        -250.0  WELLS FARGO BANK-121000248   
34    PAID AGAINST EE EXPE

In [19]:
# Missing values
print(ledger_df.isnull().sum())
print(bank_df.isnull().sum())

Doc Type                39
Posting Dt               0
Doc No                   0
Cheque No             2963
Description           1709
Particulars           2145
Debit-Credit             0
Bank                    31
GL A/C                   8
GL A/c Description       0
Amount                   0
Narration                0
dtype: int64
Date              0
Unique ID        35
Bank Ref          1
Cust Ref          1
Narration         0
Narration 2       1
Opening Flag    249
Credit-Debit      0
BANK              0
Credit          216
Debit            35
Amount            0
dtype: int64


# Reconciliation by Rules

In [20]:
ledger_df['Matched'] = False
bank_df['Matched'] = False

DATE_TOLERANCE =3
AMOUNT_TOLERANCE = 0.00
FUZZY_THRESHOLD = 50

all_matches = []


In [21]:
# Helper functions for rules

def amount_match(a, b):
    return abs(a - b) <= AMOUNT_TOLERANCE

def date_exact(d1, d2):
    return d1 == d2

def date_range(d1, d2):
    return abs((d1 - d2).days) <= DATE_TOLERANCE

def narration_exact(n1, n2):
    return n1 == n2

def narration_fuzzy(n1, n2):
    if pd.isna(n1) or pd.isna(n2):
        return False
    return fuzz.partial_ratio(n1, n2) >= FUZZY_THRESHOLD

In [22]:
def match_records(rule_name, ledger_df, bank_df, condition_func):
    matches = []

    ledger_unmatched = ledger_df[~ledger_df['Matched']]
    bank_unmatched = bank_df[~bank_df['Matched']]

    for li, lrow in ledger_unmatched.iterrows():

        candidates = bank_unmatched[
            abs(bank_unmatched['Amount'] - lrow['Amount']) <= AMOUNT_TOLERANCE
        ]

        if candidates.empty:
            continue

        # Apply rule condition
        candidates = candidates[
            candidates.apply(lambda brow: condition_func(lrow, brow), axis=1)
        ]

        if not candidates.empty:
            bi = candidates.index[0]

            ledger_df.at[li, 'Matched'] = True
            bank_df.at[bi, 'Matched'] = True

            matches.append({
                "Ledger_Index": li,
                "Bank_Index": bi,
                "Rule": rule_name,
                "Ledger_Amount": lrow['Amount'],
                "Bank_Amount": bank_df.loc[bi, 'Amount']
            })

    return matches

In [32]:
# Defining Rules in Sequence
# 1. Narration Exact + Partial
all_matches += match_records(
    "Narration Exact+Fuzzy + Amount",
    ledger_df, bank_df,
    lambda l, b: (narration_exact(l['Narration'], b['Narration']) or
                  narration_fuzzy(l['Narration'], b['Narration']))
)
# 2. Narration Exact
all_matches += match_records(
    "Narration Exact + Amount",
    ledger_df, bank_df,
    lambda l, b: narration_exact(l['Narration'], b['Narration'])
)

# 3. Date Exact
all_matches += match_records(
    "Date Exact + Amount",
    ledger_df, bank_df,
    lambda l, b: date_exact(l['Posting Dt'], b['Date'])
)

# 4. Date Range (+/- 3 days)
all_matches += match_records(
    "Date Range + Amount",
    ledger_df, bank_df,
    lambda l, b: date_range(l['Posting Dt'], b['Date'])
)

# 5. Narration Exact + Date Exact
all_matches += match_records(
    "Narration Exact + Date Exact + Amount",
    ledger_df, bank_df,
    lambda l, b: narration_exact(l['Narration'], b['Narration']) and
                 date_exact(l['Posting Dt'], b['Date'])
)

# 6. Narration Exact + Date Range
all_matches += match_records(
    "Narration Exact + Date Range + Amount",
    ledger_df, bank_df,
    lambda l, b: narration_exact(l['Narration'], b['Narration']) and
                 date_range(l['Posting Dt'], b['Date'])
)

# 7. Narration (Exact+Fuzzy) + Date (Exact + Range) +Amount
all_matches += match_records(
    "Narration Fuzzy + Date Flexible + Amount",
    ledger_df, bank_df,
    lambda l, b: (narration_exact(l['Narration'], b['Narration']) or
                  narration_fuzzy(l['Narration'], b['Narration'])) and
                 (date_exact(l['Posting Dt'], b['Date']) or
                  date_range(l['Posting Dt'], b['Date']))
)

In [33]:
# Collect all the results
recon_df = pd.DataFrame(all_matches)

In [34]:
recon_df

,Ledger_Index,Bank_Index,Rule,Ledger_Amount,Bank_Amount
0,1,14,Narration Exact+Fuzzy + Amount,-200.00,-200.00
1,10,55,Narration Exact+Fuzzy + Amount,-2065.10,-2065.10
2,11,168,Narration Exact+Fuzzy + Amount,-76.67,-76.67
3,19,23,Narration Exact+Fuzzy + Amount,-500.00,-500.00
4,21,62,Narration Exact+Fuzzy + Amount,-400.00,-400.00
...,...,...,...,...,...
169,1930,219,Date Exact + Amount,-413.38,-413.38
170,2759,3,Date Exact + Amount,-9550.73,-9550.73
171,2760,4,Date Exact + Amount,-5120.00,-5120.00
172,2761,5,Date Exact + Amount,-2465.98,-2465.98


In [26]:
all_matches

[{'Ledger_Index': 1,
  'Bank_Index': np.int64(14),
  'Rule': 'Narration Exact+Fuzzy + Amount',
  'Ledger_Amount': -200.0,
  'Bank_Amount': np.float64(-200.0)},
 {'Ledger_Index': 10,
  'Bank_Index': np.int64(55),
  'Rule': 'Narration Exact+Fuzzy + Amount',
  'Ledger_Amount': -2065.1,
  'Bank_Amount': np.float64(-2065.1)},
 {'Ledger_Index': 11,
  'Bank_Index': np.int64(168),
  'Rule': 'Narration Exact+Fuzzy + Amount',
  'Ledger_Amount': -76.67,
  'Bank_Amount': np.float64(-76.67)},
 {'Ledger_Index': 19,
  'Bank_Index': np.int64(23),
  'Rule': 'Narration Exact+Fuzzy + Amount',
  'Ledger_Amount': -500.0,
  'Bank_Amount': np.float64(-500.0)},
 {'Ledger_Index': 21,
  'Bank_Index': np.int64(62),
  'Rule': 'Narration Exact+Fuzzy + Amount',
  'Ledger_Amount': -400.0,
  'Bank_Amount': np.float64(-400.0)},
 {'Ledger_Index': 22,
  'Bank_Index': np.int64(14),
  'Rule': 'Narration Exact+Fuzzy + Amount',
  'Ledger_Amount': -200.0,
  'Bank_Amount': np.float64(-200.0)},
 {'Ledger_Index': 27,
  'Bank_In

In [27]:
match_summary = recon_df['Rule'].value_counts().reset_index()
match_summary.columns = ['Rule','Count']

In [28]:
print(match_summary)

                             Rule  Count
0  Narration Exact+Fuzzy + Amount    138
1             Date Exact + Amount     36


In [35]:
unmatched_ledger = ledger_df[~ledger_df['Matched']].copy()
unmatched_bank = bank_df[~bank_df['Matched']].copy()